In [1]:
#MILESTONE - 1
import os
import string
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)
from sklearn.metrics.pairwise import cosine_similarity


# Load dataset
def locate_train_file():
    for root, _, files in os.walk("/kaggle/input"):
        if "train.csv" in files:
            return os.path.join(root, "train.csv")
    raise FileNotFoundError("train.csv not found")

train_path = locate_train_file()
train = pd.read_csv(train_path)

print(f"Dataset loaded from: {train_path}")
print(f"Shape: {train.shape}")


# Text preprocessing helper
translator = str.maketrans("", "", string.punctuation)

def clean_text(text):
    return str(text).lower().translate(translator)


# Q1 - Answer distribution
answer_distribution = train["answer"].value_counts()

q1 = (
    answer_distribution.max()
    + answer_distribution.min()
)


# Q2 - Vocabulary size from prompts
vocabulary = set()

for prompt in train["prompt"]:
    vocabulary.update(
        clean_text(prompt).split()
    )

q2 = len(vocabulary)


# Q3 - Stopword filtering for Row ID 1
row = train.loc[train["id"] == 1]

if len(row) == 0:
    row = train.iloc[[0]]

row = row.iloc[0]

tokens = clean_text(row["prompt"]).split()

filtered_tokens = [
    word
    for word in tokens
    if word not in ENGLISH_STOP_WORDS
]

q3 = len(filtered_tokens)


# Q4 - TF-IDF feature space
combined_text = (
    train["prompt"].fillna("").astype(str)
    + " " + train["A"].fillna("").astype(str)
    + " " + train["B"].fillna("").astype(str)
    + " " + train["C"].fillna("").astype(str)
    + " " + train["D"].fillna("").astype(str)
    + " " + train["E"].fillna("").astype(str)
)

tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(combined_text)

q4 = len(tfidf.get_feature_names_out())


# Q5 - Similarity between prompt and option A
prompt_vector = tfidf.transform(
    [str(row["prompt"])]
)

option_a_vector = tfidf.transform(
    [str(row["A"])]
)

q5 = cosine_similarity(
    prompt_vector,
    option_a_vector
)[0, 0]


# Q6 - Highest similarity accuracy
labels = ["A", "B", "C", "D", "E"]

correct_predictions = 0

for _, sample in train.iterrows():

    p_vec = tfidf.transform(
        [str(sample["prompt"])]
    )

    similarities = []

    for label in labels:

        option_vec = tfidf.transform(
            [str(sample[label])]
        )

        similarities.append(
            cosine_similarity(
                p_vec,
                option_vec
            )[0, 0]
        )

    predicted_answer = labels[
        np.argmax(similarities)
    ]

    if predicted_answer == sample["answer"]:
        correct_predictions += 1

q6 = (
    correct_predictions
    / len(train)
    * 100
)


# Q7 and Q8 {formula used here is MAP@3= 1/rank}

q7 = 1.0
q8 = 0.5


# MAP@3 helper
def map_at_3(actual, predictions):

    for rank, prediction in enumerate(
        predictions[:3],
        start=1
    ):
        if prediction == actual:
            return 1 / rank

    return 0.0


# Q9 - Majority class baseline
top_three_answers = list(
    answer_distribution.index[:3]
)

baseline_scores = [
    map_at_3(answer, top_three_answers)
    for answer in train["answer"]
]

q9 = np.mean(baseline_scores)


# Q10 - TF-IDF ranking pipeline
pipeline_scores = []

for _, sample in train.iterrows():

    p_vec = tfidf.transform(
        [str(sample["prompt"])]
    )

    similarity_scores = {}

    for label in labels:

        option_vec = tfidf.transform(
            [str(sample[label])]
        )

        similarity_scores[label] = (
            cosine_similarity(
                p_vec,
                option_vec
            )[0, 0]
        )

    ranked_options = sorted(
        similarity_scores,
        key=similarity_scores.get,
        reverse=True
    )

    pipeline_scores.append(
        map_at_3(
            sample["answer"],
            ranked_options[:3]
        )
    )

q10 = np.mean(pipeline_scores)


# Results
results = pd.DataFrame(
    {
        "Question": [
            "Q1","Q2","Q3","Q4","Q5",
            "Q6","Q7","Q8","Q9","Q10"
        ],
        "Answer": [
            q1,
            q2,
            q3,
            q4,
            round(q5, 4),
            round(q6, 4),
            q7,
            q8,
            round(q9, 6),
            round(q10, 6)
        ]
    }
)

print("\nMilestone 1 Results\n")
print(results.to_string(index=False))

Dataset loaded from: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
Shape: (2000, 8)

Milestone 1 Results

Question      Answer
      Q1  814.000000
      Q2  859.000000
      Q3   13.000000
      Q4 2762.000000
      Q5    0.272000
      Q6   13.550000
      Q7    1.000000
      Q8    0.500000
      Q9    0.421250
     Q10    0.296167
